# Python 教师聊天机器人 —— 项目第 1 天

用本地 **Ollama（OpenAI 兼容接口）** 做一个「有点个性」的 Python 老师：你提问，它用 Markdown 讲清楚。


## 环境准备

### Python

- Python 3.8+（推荐 3.10+）

### Python 包

可用 pip 安装（示例）：

```bash
pip install requests python-dotenv beautifulsoup4 ipython openai
```

或使用 `requirements.txt`（按你项目里实际文件为准），常见依赖包括：`requests`、`python-dotenv`、`beautifulsoup4`、`ipython`、`openai`。

### 环境 / 外部服务

- **本地 LLM 服务器（OpenAI 兼容）**
  - 本笔记本会创建：`OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')`，并设 `MODEL='qwen3:30b-a3b'`
  - 请确保兼容服务（如 Ollama 或其它 OpenAI 兼容代理）已在该 `base_url` 运行，且模型 `qwen3:30b-a3b` 可用
  - 若主机或模型不同，请改客户端初始化与 `MODEL` 变量

- **API 密钥 / `.env`（最佳实践）**
  - 笔记本导入了 `dotenv`，但当前示例把 `api_key` 写成字面量 `'ollama'`（本地占位，不是云端密钥）
  - 建议用 `.env` 管理：`OLLAMA_BASE_URL`（默认 `http://localhost:11434/v1`）、`OLLAMA_API_KEY` / `OPENAI_API_KEY`
  - **不要**把真实 API 密钥提交进版本库


In [ ]:
# ========== 导入：本地 OpenAI 兼容客户端 +（预留）网页抓取工具 ==========

# 标准库 os：以后若改读环境变量会用到
import os
# requests：HTTP 客户端（本练习主路径未必用到，但常与抓取示例一起导入）
import requests
# load_dotenv：可从 .env 加载配置（当前格未强制调用）
from dotenv import load_dotenv
# BeautifulSoup：HTML 解析（为扩展抓取课留着）
from bs4 import BeautifulSoup
# IPython：用 Markdown 漂亮展示模型回答
from IPython.display import Markdown, display
# OpenAI SDK：即使连的是本地 Ollama，也用同一套 chat.completions API
from openai import OpenAI


In [ ]:
# ========== 客户端与模型：指到本机 Ollama 的 OpenAI 兼容端口 ==========

# base_url 指向本地 /v1；api_key='ollama' 是本地占位字符串（Ollama 通常不校验，但 SDK 要求有值）
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
# MODEL 必须和本机已拉取的模型名一致；改完请重新运行本格及后续格
MODEL = 'qwen3:30b-a3b'


In [ ]:
# ========== 系统提示词：定义「有趣的小学 Python 老师」人设 ==========

# system_prompt 整段保持英文：这是发给模型的指令，翻译会改变语气、篇幅与行为
# 字数建议约 100–200；改完后务必重新运行本格，再跑后面的 messages / create
system_prompt = '''
You are a fun python teacher. You teach the 3rd grade class, full of little students. User is amongst one of them and is a total beginner, hahaha.
But he's determined and want to increase his skills to intermediate level. User will ask you about python problems and you will try to give 
them best time complexity results according to there criteria (if a student asking a teacher, and teacher explaining him the topic like situation).

Personality: (*: People can add more personality attributes as they like. Follow the personality criteria too.)
- Comedic
- British
- 4th wall break
- Mysterious
- Dark
*<------Add Personality attributes--------->

- Shorten your word limit to be under 100-200 words, that would be appreciated.
- Use proper formatting and cell output 
- You can ask user to can I continue or ask a random student the same student then the student will do a roleplay of answering the question and you can go on this conversation if you want.
- When explaining about code please be as through as possible. 
- Remember user is a beginner so try to use easy to understand words.
- Your goal is to make the user a intermediate level python expert.

Note:- Convert your explaination into markdown everytime you answer.
'''


In [ ]:
# ========== 用户首条消息：改这里就能换开场白 ==========

# user_prompt 保持英文（或你想问的原文）：会原样进入 messages，影响模型回答
# 称呼可改成 Mam / Sir 等；改完从本格往下重新执行
user_prompt = """
Hi Mam, Can you introduce yourself?
"""


In [ ]:
# ========== 组装 messages：system 定角色，user 放问题 ==========

# Chat Completions 标准格式：列表里每项有 role + content
MESSAGES = [
        {"role":"system","content":system_prompt},
        {"role":"user", "content":user_prompt}
    ]


In [ ]:
# ========== 调用本地模型并展示 Markdown 回答 ==========

# openai.chat.completions.create：同步请求；返回 choices[0].message.content
response = openai.chat.completions.create(model=MODEL, messages=MESSAGES)
# 把模型输出当 Markdown 渲染（system_prompt 要求它用 markdown 回答）
display(Markdown(response.choices[0].message.content))


## 故障排除

- **ModuleNotFoundError**：对缺失包装 `pip install ...`
- **base_url 连接错误**：确认本地 LLM 服务在跑，且 `base_url` 一致
- **模型未找到**：把 `MODEL` 改成服务器上已有的名字（如 `ollama list`）
- **敏感数据**：密钥放 `.env`，不要写进公开仓库
